In [2]:
import os
import groq
import json
import time 
from dotenv import load_dotenv
from groq import Groq


In [3]:
# Load .env file
load_dotenv()

True

In [5]:
#check the loading of the API key
groqapikey = os.getenv("GROQ_API_KEY")
print(groqapikey[:10])

gsk_4mVLlF


In [6]:
# Create client
client = Groq(api_key=groqapikey)
print("client created sucessfully")


client created sucessfully


In [7]:
response = client.chat.completions.create(
    model = "openai/gpt-oss-120b",
    messages = [
        {"role":"user", "content": "What is the current weather of Hyderabad?"}
    ]
)
print(response.choices[0])
print(response.choices[0].message.content)

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I don’t have the ability to look up live weather data, so I can’t tell you the exact conditions in Hyderabad right now. For the most up‑to‑date forecast, you can check any of the following sources:\n\n- **Weather websites**:\u202f[Weather.com](https://weather.com), [AccuWeather](https://www.accuweather.com), or the Indian Meteorological Department’s portal.\n- **Mobile apps**:\u202fThe Weather Channel, AccuWeather, Weather Underground, or the built‑in weather widget on iOS/Android.\n- **Voice assistants**:\u202fAsk Siri, Google Assistant, or Alexa for “current weather in Hyderabad.”\n\n---\n\n### Typical Climate in Hyderabad (for reference)\n\n| Season | Average Temperatures (°C) | Common Conditions |\n|--------|---------------------------|-------------------|\n| **Winter (Dec‑Feb)** | 14\u202f–\u202f29 | Cool mornings, warm afternoons; occasional drizzle. |\n| **Summer (Mar‑May)** | 25\u202f–\u

In [8]:
response = client.chat.completions.create(
    model = "openai/gpt-oss-120b",  # You can change model
    messages = [
        {"role":"user", "content": "What is the capital of Inida?"}
    ]
)
print(response.choices[0])
print(response.choices[0].message.content)

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of India is **New\u202fDelhi**.', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user asks "What is the capital of Inida?" There\'s a typo: "Inida". Likely they meant "India". The capital of India is New Delhi. Provide answer. No policy issues.', tool_calls=None))
The capital of India is **New Delhi**.


In [9]:
response = client.chat.completions.create(
    model = "openai/gpt-oss-120b",  # You can change model
    messages = [
        {"role":"user", "content": "what is the current ranking of IIT DHANBAD "}
    ]
)
print(response.choices[0])
print(response.choices[0].message.content)

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='## Latest\u202fofficial rankings for **IIT\u202f(ISM)\u202fDhanbad** (as of\u202f2026)\n\n| Ranking Body | Year (latest released) | Programme / Category | Position / Band | Notes |\n|--------------|-----------------------|----------------------|----------------|-------|\n| **NIRF\u202f(India)** – Engineering | **2023** (released\u202fJuly\u202f2023) | Engineering (overall) | **23\u202f‑\u202f24** (tied) | 2024\u202fNIRF data are still being compiled; the 2023 list is the most recent official Indian‑government ranking. |\n| **NIRF\u202f(India)** – Overall | **2023** | Overall (all streams) | **29** | Considers teaching, learning & resources, research, graduation outcomes, outreach & perception. |\n| **QS\u202fWorld University Rankings** – Asia | **2024** (released\u202fMay\u202f2024) | Overall (Asia) | **301‑350** band | QS ranks by academic reputation, employer reputation, faculty‑student ratio,

In [11]:
#if you want the fake function that should give the weather condition (you can also call the API here to fetch the actual weather forecast\)
def get_weather(city: str):
  ''' Get weather for a city '''  #DESCRIPTION
  fake_weather = {
      "hyderabad": "25 Degrees, partly cloudy, Humidity: 65%",
      "kakinada": "35 Degrees, partly sunny, Humidity: 80%",
      "delhi": "32 Degrees, partly cloudy, Humidity: 45%",
      "pune": "19 Degrees, partly rainy, Humidity: 63%",
      "amsterdam": "23 Degrees, partly cloudy, Humidity: 40%",
      "rajahmundry": "37 Degrees, Sunny, Humidity: 89%",

  }

  city_lower = city.lower().strip()
  if city_lower in fake_weather:
    return fake_weather[city_lower]
  return f"Weather data not avaiablaes for the {city}"

print("Weather ", get_weather("dhanbad"))


Weather  Weather data not avaiablaes for the dhanbad


In [12]:
# now creating a function to calculate the mathematical expressions 
def calculate(expression: str) -> str:
    """Evaluate a math expression and return the result."""
    try:
        # Only allow safe math operations
        allowed = set('0123456789+-*/.() ')
        if not all(c in allowed for c in expression):
            return "Error: Only numbers and +-*/() allowed"
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"


In [16]:
#till now we have created a functions and now we will register these functions in JSON schema 
#we register these tool functions in the json format som that the llm can understand the functions and use the tool accordingly


In [15]:
tools = [

    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city, e.g. Hyderabad"
                    }
                },
                "required": ["city"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression. Use this for mathematical computations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The mathematical expression to evaluate, e.g. '(25 * 4) + 100'"
                    }
                },
                "required": ["expression"]
            }
        }
    }

]

print("Tools are defined in a json format")

for tool in tools:
    name = tool["function"]["name"]
    desc = tool["function"]["description"]
    params = list(tool["function"]["parameters"]["properties"].keys())

    print(f'{name} ({" ".join(params)}) - {desc}')

Tools are defined in a json format
get_weather (city) - Get the current weather for a city.
calculate (expression) - Evaluate a mathematical expression. Use this for mathematical computations.


In [18]:
#configure the LLM to use the tools 
messages = [
    {"role":"system", "content": "You are a helpful assistant and use tools when needed!"},
    {"role": "user", "content": "What is the weather of Hyderabad?"}
]

response = client.chat.completions.create(
    model = "openai/gpt-oss-120b",
    messages = messages,
    tools = tools
)

# Let's inspect what the LLM decided
choice = response.choices[0]
print(f"Finish reason: {choice.finish_reason}")
print(f"Content: {choice.message.content}")
print(f"Tool calls: {choice.message.tool_calls}")

if choice.message.tool_calls:
    tc = choice.message.tool_calls[0]
    print(f"\n── LLM Decision ──")
    print(f"  Tool:      {tc.function.name}")
    print(f"  Arguments: {tc.function.arguments}")
    print(f"  ID:        {tc.id}")

Finish reason: tool_calls
Content: None
Tool calls: [ChatCompletionMessageToolCall(id='fc_00ec4e47-8194-4c44-9da8-f77a338ddc90', function=Function(arguments='{"city":"Hyderabad"}', name='get_weather'), type='function')]

── LLM Decision ──
  Tool:      get_weather
  Arguments: {"city":"Hyderabad"}
  ID:        fc_00ec4e47-8194-4c44-9da8-f77a338ddc90


In [19]:
messages = [
    {"role":"system", "content": "You are a helpful assistant and use tools when needed!"},
    {"role": "user", "content": "What is the weather of Hyderabad?"}
]

response = client.chat.completions.create(
    model = "openai/gpt-oss-120b",
    messages = messages,
    tools = tools
)

# Let's inspect what the LLM decided
choice = response.choices[0]
print(f"Finish reason: {choice.finish_reason}")
print(f"Content: {choice.message.content}")
print(f"Tool calls: {choice.message.tool_calls}")

if choice.message.tool_calls:
    tc = choice.message.tool_calls[0]
    print(f"\n── LLM Decision ──")
    print(f"  Tool:      {tc.function.name}")
    print(f"  Arguments: {tc.function.arguments}")
    print(f"  ID:        {tc.id}")


available_tools = {
    "calculate": calculate,
    "get_weather": get_weather
}

tool_call = choice.message.tool_calls[0]
print(tool_call)

function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)
print(function_name, arguments)

result = available_tools[function_name](**arguments)
print(result)


messages.append(choice.message)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result
})

final_respose = client.chat.completions.create(
    model = "openai/gpt-oss-120b",
    messages = messages,
    tools = tools
)
print(final_respose.choices[0].message.content)

Finish reason: tool_calls
Content: None
Tool calls: [ChatCompletionMessageToolCall(id='fc_669fcac9-418d-4140-acf9-96a2e3343e8a', function=Function(arguments='{"city":"Hyderabad"}', name='get_weather'), type='function')]

── LLM Decision ──
  Tool:      get_weather
  Arguments: {"city":"Hyderabad"}
  ID:        fc_669fcac9-418d-4140-acf9-96a2e3343e8a
ChatCompletionMessageToolCall(id='fc_669fcac9-418d-4140-acf9-96a2e3343e8a', function=Function(arguments='{"city":"Hyderabad"}', name='get_weather'), type='function')
get_weather {'city': 'Hyderabad'}
25 Degrees, partly cloudy, Humidity: 65%
The current weather in Hyderabad is **25 °C**, partly cloudy, with a humidity of about **65 %**.


In [20]:
# ──────────────────────────────────────
# THE AGENT LOOP — The core of every agent
# ──────────────────────────────────────

def run_agent(user_message, max_steps=5, verbose=True):
    """
    Run a simple agent that can use tools to answer questions.

    Args:
        user_message: The user's question
        max_steps: Hard stop — max tool calls before giving up
        verbose: Print each step for debugging
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the available tools to answer questions accurately. If you can answer without tools, do so directly."},
        {"role": "user", "content": user_message}
    ]

    if verbose:
        print(f"\n{'═' * 60}")
        print(f"🧑 User: {user_message}")
        print(f"{'═' * 60}")

    for step in range(max_steps):
        # Ask the LLM
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages,
            tools=tools,
        )

        choice = response.choices[0]

        # Case 1: LLM is done — it has a final answer
        if choice.finish_reason == "stop":
            if verbose:
                print(f"\n✅ Step {step + 1}: Final answer ready")
            return choice.message.content

        # Case 2: LLM wants to use a tool
        if choice.message.tool_calls:
            # Add assistant message to history
            messages.append(choice.message)

            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"\n🔧 Step {step + 1}: Calling {func_name}({args})")

                # Run the tool
                if func_name in available_tools:
                    result = available_tools[func_name](**args)
                else:
                    result = f"Error: Unknown tool '{func_name}'"

                if verbose:
                    print(f"   📤 Result: {result}")

                # Send result back
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

    # Hard stop reached
    return "⚠️ Agent reached maximum steps without finishing."

print("✅ Agent function defined!")

✅ Agent function defined!


In [21]:
# Test 2: Weather question (needs weather tool)
answer = run_agent("What's the weather like in Pune?")
print(f"\n🤖 Answer: {answer}")


════════════════════════════════════════════════════════════
🧑 User: What's the weather like in Pune?
════════════════════════════════════════════════════════════

🔧 Step 1: Calling get_weather({'city': 'Pune'})
   📤 Result: 19 Degrees, partly rainy, Humidity: 63%

✅ Step 2: Final answer ready

🤖 Answer: The current weather in Pune is **19 °C**, partly rainy, with a humidity of about **63 %**.


In [22]:
# Test 2: Weather question (needs weather tool)
answer = run_agent("What's the (45*67)+ (23*89)")
print(f"\n🤖 Answer: {answer}")


════════════════════════════════════════════════════════════
🧑 User: What's the (45*67)+ (23*89)
════════════════════════════════════════════════════════════

🔧 Step 1: Calling calculate({'expression': '(45*67)+(23*89)'})
   📤 Result: 5062

✅ Step 2: Final answer ready

🤖 Answer: The result of \((45 \times 67) + (23 \times 89)\) is **5,062**.
